# 🐻 Gấu Học Vui — Tạo giọng cô giáo bằng OmniVoice

Sổ này dùng mô hình **OmniVoice** (k2-fsa, giấy phép Apache-2.0) để đọc lại toàn bộ 666 câu trong app với giọng tự nhiên hơn.

**Cách dùng**
1. Menu **Runtime › Change runtime type** › chọn **T4 GPU** › Save.
2. Chạy lần lượt từng ô (bấm ▶ bên trái mỗi ô), đọc chữ hướng dẫn ở mỗi ô.
3. Ô cuối tải về file **gau-voice-pack.zip** → gửi file đó cho Claude trong cuộc trò chuyện để cập nhật app.

Nếu bị ngắt giữa chừng, chạy lại các ô: câu đã đọc xong được giữ lại, không phải làm từ đầu.

In [ ]:
#@title 1. Kiểm tra card đồ họa và cài OmniVoice (2–4 phút)
import subprocess, torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "KHÔNG CÓ — vào Runtime › Change runtime type › T4 GPU")
!pip install -q omnivoice soundfile
!ffmpeg -version | head -1
print("✓ Cài xong")

In [ ]:
%%writefile build_voice_pack.py
"""
Tạo gói giọng đọc cho Gấu Học Vui.

Hai bộ máy đọc (chọn bằng --engine):
  • omnivoice  — OmniVoice (k2-fsa, Apache-2.0): giọng tự nhiên, cần card đồ họa (Colab T4, GTX 1660…).
  • piper      — Piper: chạy trên CPU, nhẹ, giọng máy hơn. Dùng làm dự phòng.

Đầu vào : scripts/voice-texts.json  (tạo bằng: npm run voice:texts)
Đầu ra  : public/audio/<nhóm>.mp3 + public/audio/manifest.json

Mỗi nhóm (1 chủ đề / 1 bài thơ) gộp thành 1 file mp3 ("sprite"); manifest ghi câu nào nằm ở giây
thứ mấy. Thơ/truyện có thêm mốc thời gian từng chữ để chạy karaoke. App không cần sửa gì khi đổi giọng.

Ví dụ:
  # OmniVoice, sao chép giọng từ một đoạn ghi âm 5–10 giây (khuyên dùng)
  python scripts/build_voice_pack.py --engine omnivoice --ref-audio me.wav --ref-text "Chào con yêu! ..."
  # OmniVoice, tự thiết kế giọng cô giáo (không cần ghi âm)
  python scripts/build_voice_pack.py --engine omnivoice --instruct "female, young adult, high pitch"
  # Piper
  python scripts/build_voice_pack.py --engine piper --model vi_VN-vais1000-medium.onnx
Cần ffmpeg trong PATH để nén mp3.
"""
import argparse, hashlib, json, re, subprocess, sys, tempfile, time, unicodedata, wave
from pathlib import Path

import numpy as np

ROOT = Path(__file__).resolve().parent.parent
PRE_PAD, POST_PAD, GAP = 0.06, 0.06, 0.25   # giây
PUNCT_PAUSE = 0.18                            # trọng số nghỉ sau dấu câu khi chia thời gian chữ

# Câu dùng để tạo giọng mẫu khi thiết kế giọng (không có ghi âm).
SEED_TEXT = "Chào con yêu! Hôm nay cô cùng con đọc thơ, đếm số và tìm các bạn con vật nhé. Con sẵn sàng chưa nào?"


def norm(t: str) -> str:
    """Phải khớp normKey() trong src/services/audio/voicePack.ts"""
    return re.sub(r"\s+", " ", unicodedata.normalize("NFC", t).strip())


def trim_silence(a: np.ndarray, sr: int) -> np.ndarray:
    if a.size == 0:
        return a
    win = max(1, int(sr * 0.01))
    env = np.sqrt(np.convolve(a.astype(np.float32) ** 2, np.ones(win) / win, mode="same"))
    idx = np.where(env > max(250.0, env.max() * 0.04))[0]
    return a[idx[0]: idx[-1] + 1] if idx.size else a


# ─────────────────────────── Bộ máy đọc ───────────────────────────

class PiperEngine:
    name = "piper"

    def __init__(self, a):
        from piper import PiperVoice
        self.v = PiperVoice.load(a.model)
        self.sr = self.v.config.sample_rate
        self.voice_id = Path(a.model).stem
        self.scale = {"card": a.length_scale, "story": a.story_length_scale}
        self.cache: dict = {}

    def synth(self, texts: list[str], kind: str) -> list[np.ndarray]:
        from piper import SynthesisConfig
        out = []
        for t in texts:
            key = (t, kind)
            if key not in self.cache:
                parts = [c.audio_int16_array for c in self.v.synthesize(t, SynthesisConfig(length_scale=self.scale[kind]))]
                self.cache[key] = np.concatenate(parts) if parts else np.zeros(0, np.int16)
            out.append(self.cache[key])
        return out

    def word_weight(self, word: str, kind: str) -> float:
        """Độ dài khi đọc riêng chữ đó — dùng để chia mốc karaoke."""
        clean = re.sub(r"[^\w]", "", word)
        return len(trim_silence(self.synth([clean], kind)[0], self.sr)) / self.sr if clean else 0.05


class OmniVoiceEngine:
    name = "omnivoice"

    def __init__(self, a, cache_dir: Path):
        import torch
        from omnivoice import OmniVoice
        dev = a.device or ("cuda:0" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
        dtype = torch.float16 if dev.startswith("cuda") else torch.float32
        print(f"Đang tải mô hình {a.omni_model} lên {dev}…")
        self.m = OmniVoice.from_pretrained(a.omni_model, device_map=dev, dtype=dtype)
        self.sr = int(getattr(self.m, "sampling_rate", 24000))
        self.lang = a.language
        self.steps = a.num_step
        self.batch = a.batch_size
        self.speed = {"card": a.speed, "story": a.story_speed}
        self.cache_dir = cache_dir
        self.prompt = self._voice_prompt(a)
        self.voice_id = f"omnivoice:{Path(a.ref_audio).stem if a.ref_audio else a.instruct}"

    def _voice_prompt(self, a):
        """Một giọng duy nhất cho cả app: sao chép từ ghi âm, hoặc thiết kế 1 lần rồi sao chép lại."""
        from omnivoice import VoiceClonePrompt
        saved = self.cache_dir / "voice_prompt.pt"
        if saved.exists() and not a.new_voice:
            print("Dùng lại giọng đã tạo lần trước:", saved)
            return VoiceClonePrompt.load(str(saved))
        if a.ref_audio:
            prompt = self.m.create_voice_clone_prompt(ref_audio=a.ref_audio, ref_text=a.ref_text or None)
        else:
            print(f'Thiết kế giọng: "{a.instruct}"')
            seed = self.m.generate(text=SEED_TEXT, language=self.lang, instruct=a.instruct, num_step=self.steps)[0]
            import soundfile as sf
            seed_wav = self.cache_dir / "voice_seed.wav"
            sf.write(seed_wav, seed, self.sr)
            print("Giọng mẫu (nghe thử trước khi chạy hết):", seed_wav)
            prompt = self.m.create_voice_clone_prompt(ref_audio=str(seed_wav), ref_text=SEED_TEXT)
        prompt.save(str(saved))
        return prompt

    def _key(self, text: str, kind: str) -> Path:
        h = hashlib.sha1(f"{text}|{kind}|{self.speed[kind]}|{self.steps}".encode()).hexdigest()[:16]
        return self.cache_dir / "clips" / f"{h}.npy"

    def synth(self, texts: list[str], kind: str) -> list[np.ndarray]:
        (self.cache_dir / "clips").mkdir(parents=True, exist_ok=True)
        todo = [t for t in dict.fromkeys(texts) if not self._key(t, kind).exists()]
        for i in range(0, len(todo), self.batch):
            chunk = todo[i: i + self.batch]
            audios = self.m.generate(text=chunk, language=[self.lang] * len(chunk), voice_clone_prompt=[self.prompt] * len(chunk),
                                     speed=self.speed[kind], num_step=self.steps)
            for t, au in zip(chunk, audios):
                pcm = np.clip(np.asarray(au, dtype=np.float32), -1, 1)
                np.save(self._key(t, kind), (pcm * 32767).astype(np.int16))
        return [np.load(self._key(t, kind)) for t in texts]

    def word_weight(self, word: str, kind: str) -> float:
        """Tiếng Việt mỗi chữ là một âm tiết: ước lượng theo số chữ cái thay vì đọc riêng (đỡ tốn GPU)."""
        clean = re.sub(r"[^\w]", "", word)
        return 0.16 + 0.035 * len(clean) if clean else 0.05


# ─────────────────────────── Karaoke ───────────────────────────

def word_timings(eng, kind: str, text: str, clip: np.ndarray) -> list[list[float]]:
    """Mốc từng chữ: chia theo độ dài ước lượng của từng chữ, rồi bắt vào chỗ lặng gần nhất."""
    sr = eng.sr
    tokens = text.split(" ")
    durs = [eng.word_weight(tok, kind) for tok in tokens]
    pauses = [PUNCT_PAUSE if re.search(r"[,.:;!?]$", tok) else 0.0 for tok in tokens]
    L = len(clip) / sr
    total = sum(durs) + sum(pauses[:-1])
    k = L / total if total else 1
    win = max(1, int(sr * 0.01))
    env = np.sqrt(np.convolve(clip.astype(np.float32) ** 2, np.ones(win) / win, mode="same"))

    def snap(t: float, reach: float) -> float:
        a, b = int(max(0, t - reach) * sr), int(min(L, t + reach) * sr)
        return (a + int(np.argmin(env[a:b]))) / sr if b > a else t

    nominal, t = [], 0.0
    for i, d in enumerate(durs):
        nominal.append(t)
        t += (d + pauses[i]) * k
    starts = [0.0]
    for i in range(1, len(nominal)):
        reach = min(0.07, 0.3 * k * min(durs[i - 1], durs[i]))
        starts.append(max(snap(nominal[i], reach), starts[-1] + 0.6 * k * durs[i - 1]))
    return [[round(st + PRE_PAD, 3), round((starts[i + 1] if i + 1 < len(starts) else L) + PRE_PAD, 3)] for i, st in enumerate(starts)]


# ─────────────────────────── Chương trình chính ───────────────────────────

def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--engine", choices=["omnivoice", "piper"], default="omnivoice")
    ap.add_argument("--texts", default=str(ROOT / "scripts/voice-texts.json"))
    ap.add_argument("--out", default=str(ROOT / "public/audio"))
    ap.add_argument("--cache", default=str(ROOT / ".voice-cache"), help="thư mục lưu câu đã đọc để chạy tiếp khi bị ngắt")
    ap.add_argument("--bitrate", default="48k")
    ap.add_argument("--only", help="chỉ làm các nhóm có id chứa chuỗi này (vd: story-tho-yeu-me) để nghe thử")
    # OmniVoice
    ap.add_argument("--omni-model", default="k2-fsa/OmniVoice")
    ap.add_argument("--ref-audio", help="ghi âm giọng mẫu 5–10 giây (.wav/.mp3/.m4a)")
    ap.add_argument("--ref-text", help="lời đọc đúng trong ghi âm (bỏ trống thì cần Whisper tự nghe)")
    ap.add_argument("--instruct", default="female, young adult, high pitch", help="mô tả giọng khi không có ghi âm")
    ap.add_argument("--new-voice", action="store_true", help="tạo giọng mới, bỏ giọng đã lưu")
    ap.add_argument("--language", default="Vietnamese")
    ap.add_argument("--speed", type=float, default=0.85, help="tốc độ đọc thẻ, <1 là chậm hơn")
    ap.add_argument("--story-speed", type=float, default=0.8, help="tốc độ đọc thơ/truyện")
    ap.add_argument("--num-step", type=int, default=32)
    ap.add_argument("--batch-size", type=int, default=8)
    ap.add_argument("--device")
    # Piper
    ap.add_argument("--model", help="file .onnx của Piper")
    ap.add_argument("--length-scale", type=float, default=1.5)
    ap.add_argument("--story-length-scale", type=float, default=1.8)
    ap.add_argument("--tempo", type=float, help="kéo giãn thêm (giữ cao độ), <1 là chậm hơn. Mặc định: piper 0.8, omnivoice 1.0")
    ap.add_argument("--story-tempo", type=float)
    a = ap.parse_args()

    cache = Path(a.cache); cache.mkdir(parents=True, exist_ok=True)
    if a.engine == "piper":
        if not a.model:
            sys.exit("Piper cần --model đường/dẫn/vi_VN-vais1000-medium.onnx")
        eng = PiperEngine(a)
        tempo = {"card": a.tempo or 0.8, "story": a.story_tempo or 0.78}
    else:
        eng = OmniVoiceEngine(a, cache)
        tempo = {"card": a.tempo or 1.0, "story": a.story_tempo or 1.0}

    jobs = json.loads(Path(a.texts).read_text(encoding="utf8"))
    if a.only:
        jobs = [j for j in jobs if a.only in j["id"]]
    out = Path(a.out); out.mkdir(parents=True, exist_ok=True)
    mpath = out / "manifest.json"
    # Khi chỉ làm một phần (--only) thì giữ các nhóm cũ trong manifest.
    old = json.loads(mpath.read_text(encoding="utf8")) if a.only and mpath.exists() else None
    manifest = {"voice": eng.voice_id, "generated": time.strftime("%Y-%m-%d"),
                "sprites": dict(old["sprites"]) if old else {}, "clips": dict(old["clips"]) if old else {}}
    sr = eng.sr
    pad = lambda sec: np.zeros(int(sec * sr), np.int16)
    total = sum(len(j["texts"]) for j in jobs)
    done_n, t0 = 0, time.time()

    for job in jobs:
        kind = "story" if job.get("words") else "card"
        tp = tempo[kind]
        clips = eng.synth(job["texts"], kind)
        pieces, pos, keys = [], 0, []
        for text, raw in zip(job["texts"], clips):
            clip = trim_silence(raw, sr)
            if clip.size == 0:
                print(f"  ! bỏ qua (không đọc được): {text}", file=sys.stderr); continue
            seg = np.concatenate([pad(PRE_PAD), clip, pad(POST_PAD)])
            entry = [job["id"], round(pos / sr, 3), round(len(seg) / sr, 3)]
            if job.get("words"):
                entry.append(word_timings(eng, kind, text, clip))
            manifest["clips"][norm(text)] = entry
            keys.append(norm(text))
            pieces += [seg, pad(GAP)]
            pos += len(seg) + int(GAP * sr)
        if not pieces:
            continue
        audio = np.concatenate(pieces)
        with tempfile.NamedTemporaryFile(suffix=".wav", delete=False) as tmp:
            with wave.open(tmp.name, "wb") as w:
                w.setnchannels(1); w.setsampwidth(2); w.setframerate(sr); w.writeframes(audio.tobytes())
            af = ["-af", f"atempo={tp}"] if abs(tp - 1) > 1e-3 else []
            subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-i", tmp.name, *af, "-ac", "1", "-b:a", a.bitrate, str(out / f"{job['id']}.mp3")], check=True)
        k = 1 / tp  # atempo giãn thời gian đều tay → mọi mốc nhân cùng một hệ số
        for key in keys:
            e = manifest["clips"][key]
            e[1], e[2] = round(e[1] * k, 3), round(e[2] * k, 3)
            if len(e) > 3:
                e[3] = [[round(x * k, 3), round(y * k, 3)] for x, y in e[3]]
        manifest["sprites"][job["id"]] = f"audio/{job['id']}.mp3"
        done_n += len(job["texts"])
        el = time.time() - t0
        print(f"✓ {job['id']}: {len(job['texts'])} câu, {len(audio) / sr / tp:.1f}s  [{done_n}/{total}, còn ~{el / done_n * (total - done_n) / 60:.0f} phút]", flush=True)

    mpath.write_text(json.dumps(manifest, ensure_ascii=False, separators=(",", ":")), encoding="utf8")
    print(f"Xong: {len(manifest['clips'])} câu, {len(manifest['sprites'])} file → {out}")


if __name__ == "__main__":
    main()


In [ ]:
#@title 2. Nạp danh sách câu của app
import json, pathlib
TEXTS = r'''[
 {
  "id": "ui",
  "texts": [
   "Cô giáo đang chuẩn bị thẻ cho con nhé.",
   "Ba",
   "Hai",
   "Một",
   "Bắt đầu nào!",
   "Hết bài rồi. Con giỏi lắm!",
   "Con mèo",
   "Chuẩn luôn!",
   "Tuyệt vời!",
   "Giỏi quá!",
   "Hay lắm!",
   "Đây là",
   "Mình cùng làm quen nhé!",
   "Đây là gì nhỉ?",
   "Soi kính lúp xem nào!",
   "Đúng rồi!",
   "Thử lại nhé!",
   "Bé hãy chọn đúng hình",
   "Bé tìm",
   "Bé chọn chủ đề nhé!",
   "Bé kéo hình vào tranh nhé!",
   "Xóa hết rồi, mình dán lại nào!",
   "Bé kéo mỗi bạn về đúng cái bóng nhé!",
   "Bé đếm xem có bao nhiêu",
   "Bé làm giống hình mẫu nhé!",
   "Bé xếp vòng to nhất trước nhé!",
   "Chưa đúng, vòng này nhỏ quá!",
   "Bé chạm để xoay cho đúng hình nhé!",
   "Bé kéo sợi chỉ đi theo đường chấm, xỏ qua lỗ kim nhé!",
   "Ối, chỉ tuột rồi! Thử lại nhé!",
   "Bé mặc quần áo cho gấu nhé!",
   "Bé lật hai thẻ giống nhau nhé!",
   "Bé tô theo nét chấm nhé!",
   "Bé chọn màu rồi chạm vào tranh để tô nhé!",
   "Bé vẽ gì cũng được nhé!",
   "Cô cất tranh vào bộ sưu tập rồi nhé!",
   "Bé đặt nốt",
   "lên đúng phím nhé!",
   "Bé nghe xem đây là tiếng gì?",
   "Có bao nhiêu quả trứng?",
   "Số nào còn thiếu nhỉ?",
   "Bé chọn chậu có",
   "bông hoa",
   "Bé kéo dấu vào ô trống nhé!",
   "lớn hơn",
   "bé hơn",
   "bằng",
   "Bé tập viết số nhé!",
   "Mình cùng luyện tập nhé!",
   "Mình cùng đếm nhé!",
   "Chào Gấu! Hôm nay mình học gì nhỉ?",
   "Chọn tối đa 5 chủ đề thôi nhé",
   "Giỏi quá! Gấu xem hết thẻ rồi!",
   "Giỏi quá! Gấu làm xong rồi!",
   "Không",
   "Bốn",
   "Năm",
   "Sáu",
   "Bảy",
   "Tám",
   "Chín",
   "Mười",
   "Truyện thơ tiềm thức",
   "Nghe và tìm",
   "Đại dương",
   "Rừng rậm",
   "Dán hình",
   "Nông trại",
   "Thơ",
   "Đồng dao",
   "Truyện",
   "Ca dao",
   "Tráo thẻ",
   "Nhận biết",
   "Ngôn ngữ",
   "Quan sát",
   "Vận động tinh",
   "Ghi nhớ",
   "Toán học",
   "Âm nhạc",
   "Mỹ thuật",
   "Tráo thẻ sắp có rồi, Gấu đợi nhé!",
   "Nhận biết sắp có rồi, Gấu đợi nhé!",
   "Ngôn ngữ sắp có rồi, Gấu đợi nhé!",
   "Quan sát sắp có rồi, Gấu đợi nhé!",
   "Vận động tinh sắp có rồi, Gấu đợi nhé!",
   "Ghi nhớ sắp có rồi, Gấu đợi nhé!",
   "Toán học sắp có rồi, Gấu đợi nhé!",
   "Âm nhạc sắp có rồi, Gấu đợi nhé!",
   "Mỹ thuật sắp có rồi, Gấu đợi nhé!",
   "Tìm bóng",
   "Đếm vật ẩn",
   "Làm theo mẫu",
   "Xếp tháp màu",
   "Xoay hình",
   "Xỏ chỉ qua kim",
   "Mặc áo cho gấu",
   "Lật thẻ dễ",
   "Lật thẻ vừa",
   "Lật thẻ khó",
   "Tô hình theo nét",
   "Tô màu tranh",
   "Vẽ tự do",
   "Bộ sưu tập",
   "Đàn piano",
   "Kéo nốt nhạc",
   "Tiếng gì đây?",
   "Làm quen chữ số",
   "Đếm trứng",
   "Số còn thiếu",
   "Chọn chậu hoa",
   "So sánh",
   "Cái trống",
   "Cái chuông",
   "Cái kèn",
   "Trời mưa",
   "Tiếng sấm",
   "Chim hót",
   "Đồng hồ",
   "Còi ô tô",
   "Con mèo kêu",
   "Bánh mì",
   "Rau xà lách",
   "Phô mai",
   "Cà chua",
   "Thịt nguội",
   "Trứng",
   "Kem dâu",
   "Kem vani",
   "Kem sô cô la",
   "Kem bạc hà",
   "Quả anh đào",
   "Cái mũ",
   "Cái nón",
   "Cái áo",
   "Cái váy",
   "Cái quần dài",
   "Cái quần đùi",
   "Đôi giày",
   "Đôi ủng",
   "Đô",
   "Rê",
   "Mi",
   "Pha",
   "Son",
   "La",
   "Si",
   "Đố",
   "Ngôi nhà",
   "Con cá",
   "Bông hoa",
   "Ô tô",
   "Hình tròn",
   "Hình vuông",
   "Hình tam giác",
   "Mũi tên",
   "Ngôi sao",
   "Trái tim",
   "Số không",
   "Số không giống quả trứng",
   "Số một",
   "Số một giống cây bút chì",
   "Số hai",
   "Số hai giống con vịt",
   "Số ba",
   "Số ba giống cái tai",
   "Số bốn",
   "Số bốn giống cánh buồm",
   "Số năm",
   "Số năm giống cái móc câu",
   "Số sáu",
   "Số sáu giống con ốc sên",
   "Số bảy",
   "Số bảy giống lá cờ",
   "Số tám",
   "Số tám giống người tuyết",
   "Số chín",
   "Số chín giống quả bóng bay",
   "Động vật nuôi",
   "Động vật thủy sinh",
   "Động vật hoang dã",
   "Động vật có cánh",
   "Bọ & côn trùng",
   "Trái cây",
   "Rau củ",
   "Phương tiện",
   "Đồ chơi",
   "Đồ vật trong nhà",
   "Thiên nhiên",
   "Bảng chữ cái thường",
   "Bảng chữ cái hoa",
   "Thẻ chấm",
   "Số đếm",
   "Hình 2D",
   "Màu sắc",
   "Yêu mẹ",
   "Bạn mới",
   "Đi học đúng giờ",
   "Gió",
   "Ngôi nhà của bé",
   "Cô dạy",
   "Rửa tay",
   "Ăn ngoan",
   "Chào buổi sáng",
   "Đánh răng",
   "Mưa rơi",
   "Bố của bé",
   "Trăng tròn",
   "Chú gà con",
   "Con cá vàng",
   "Đi ngủ",
   "Cầu vồng",
   "Bé tập đếm",
   "Chú bướm vàng",
   "Cất đồ chơi",
   "Công cha như núi Thái Sơn",
   "Bầu ơi thương lấy bí cùng",
   "Con kiến mà leo cành đa",
   "Trâu ơi ta bảo trâu này",
   "Con cò mà đi ăn đêm",
   "Thằng Bờm",
   "Cái bống",
   "Ông giẳng ông giăng",
   "Con gà cục tác lá chanh",
   "Dung dăng dung dẻ",
   "Con mèo mà trèo cây cau",
   "Kéo cưa lừa xẻ",
   "Nu na nu nống",
   "Bà còng đi chợ",
   "Rùa và Thỏ",
   "Gấu con và hũ mật",
   "Kiến và Bồ Câu",
   "Ngôi sao nhỏ đi ngủ"
  ]
 },
 {
  "id": "topic-pets",
  "texts": [
   "Con chó",
   "Con mèo",
   "Con ngựa",
   "Con bò sữa",
   "Con trâu",
   "Con lợn",
   "Con gà",
   "Con vịt",
   "Con dê",
   "Con cừu",
   "Con thỏ",
   "Gà tây"
  ]
 },
 {
  "id": "topic-sea",
  "texts": [
   "Con cá",
   "Cá nhiệt đới",
   "Cá nóc",
   "Cá mập",
   "Cá heo",
   "Cá voi",
   "Bạch tuộc",
   "Con cua",
   "Con tôm",
   "Con rùa",
   "Con mực",
   "Hải cẩu"
  ]
 },
 {
  "id": "topic-wild",
  "texts": [
   "Con voi",
   "Sư tử",
   "Con hổ",
   "Hươu cao cổ",
   "Ngựa vằn",
   "Con khỉ",
   "Con gấu",
   "Con cáo",
   "Chó sói",
   "Tê giác",
   "Hà mã",
   "Cá sấu"
  ]
 },
 {
  "id": "topic-birds",
  "texts": [
   "Con vẹt",
   "Cú mèo",
   "Đại bàng",
   "Chim sẻ",
   "Bồ câu",
   "Con công",
   "Hồng hạc",
   "Chim cánh cụt",
   "Thiên nga",
   "Gà trống"
  ]
 },
 {
  "id": "topic-insects",
  "texts": [
   "Con bướm",
   "Con ong",
   "Bọ rùa",
   "Con kiến",
   "Con dế",
   "Ốc sên",
   "Con nhện",
   "Con sâu",
   "Bọ cánh cứng",
   "Con muỗi"
  ]
 },
 {
  "id": "topic-fruits",
  "texts": [
   "Quả táo",
   "Quả chuối",
   "Quả cam",
   "Dưa hấu",
   "Chùm nho",
   "Dâu tây",
   "Quả xoài",
   "Quả dứa",
   "Quả dừa",
   "Quả lê"
  ]
 },
 {
  "id": "topic-vegetables",
  "texts": [
   "Củ cà rốt",
   "Bắp ngô",
   "Củ khoai tây",
   "Quả cà chua",
   "Quả dưa chuột",
   "Bông cải xanh",
   "Củ hành",
   "Củ tỏi",
   "Quả ớt",
   "Quả cà tím"
  ]
 },
 {
  "id": "topic-vehicles",
  "texts": [
   "Xe ô tô",
   "Xe buýt",
   "Xe đạp",
   "Xe máy",
   "Xe cứu thương",
   "Xe cứu hỏa",
   "Xe cảnh sát",
   "Máy bay",
   "Tàu thủy",
   "Trực thăng"
  ]
 },
 {
  "id": "topic-toys",
  "texts": [
   "Gấu bông",
   "Quả bóng",
   "Cánh diều",
   "Bóng bay",
   "Tàu hỏa",
   "Xếp hình",
   "Cái trống",
   "Xe trượt",
   "Xúc xắc",
   "Con quay yo-yo"
  ]
 },
 {
  "id": "topic-home",
  "texts": [
   "Cái giường",
   "Cái ghế",
   "Cái cửa",
   "Bồn tắm",
   "Ti vi",
   "Đồng hồ báo thức",
   "Cái chổi",
   "Bàn chải đánh răng",
   "Cái thìa",
   "Chìa khóa"
  ]
 },
 {
  "id": "topic-nature",
  "texts": [
   "Mặt trời",
   "Mặt trăng",
   "Ngôi sao",
   "Đám mây",
   "Mưa",
   "Cầu vồng",
   "Cái cây",
   "Bông hoa",
   "Sóng biển",
   "Ngọn núi"
  ]
 },
 {
  "id": "topic-vi-lower",
  "texts": [
   "chữ a",
   "chữ á",
   "chữ ớ",
   "chữ bê",
   "chữ xê",
   "chữ dê",
   "chữ đê",
   "chữ e",
   "chữ ê",
   "chữ giê",
   "chữ hát",
   "chữ i",
   "chữ ca",
   "chữ e lờ",
   "chữ em mờ",
   "chữ en nờ",
   "chữ o",
   "chữ ô",
   "chữ ơ",
   "chữ pê",
   "chữ quy",
   "chữ e rờ",
   "chữ ét xì",
   "chữ tê",
   "chữ u",
   "chữ ư",
   "chữ vê",
   "chữ ích xì",
   "chữ i dài"
  ]
 },
 {
  "id": "topic-vi-upper",
  "texts": [
   "chữ a hoa",
   "chữ á hoa",
   "chữ ớ hoa",
   "chữ bê hoa",
   "chữ xê hoa",
   "chữ dê hoa",
   "chữ đê hoa",
   "chữ e hoa",
   "chữ ê hoa",
   "chữ giê hoa",
   "chữ hát hoa",
   "chữ i hoa",
   "chữ ca hoa",
   "chữ e lờ hoa",
   "chữ em mờ hoa",
   "chữ en nờ hoa",
   "chữ o hoa",
   "chữ ô hoa",
   "chữ ơ hoa",
   "chữ pê hoa",
   "chữ quy hoa",
   "chữ e rờ hoa",
   "chữ ét xì hoa",
   "chữ tê hoa",
   "chữ u hoa",
   "chữ ư hoa",
   "chữ vê hoa",
   "chữ ích xì hoa",
   "chữ i dài hoa"
  ]
 },
 {
  "id": "topic-dots",
  "texts": [
   "Một",
   "Hai",
   "Ba",
   "Bốn",
   "Năm",
   "Sáu",
   "Bảy",
   "Tám",
   "Chín",
   "Mười"
  ]
 },
 {
  "id": "topic-digits",
  "texts": [
   "Số không",
   "Số một",
   "Số hai",
   "Số ba",
   "Số bốn",
   "Số năm",
   "Số sáu",
   "Số bảy",
   "Số tám",
   "Số chín",
   "Số mười"
  ]
 },
 {
  "id": "topic-shapes",
  "texts": [
   "Hình tròn",
   "Hình vuông",
   "Hình tam giác",
   "Hình chữ nhật",
   "Ngôi sao",
   "Trái tim",
   "Hình thoi",
   "Hình lục giác",
   "Hình bầu dục",
   "Hình ngũ giác"
  ]
 },
 {
  "id": "topic-colors",
  "texts": [
   "Màu đỏ",
   "Màu cam",
   "Màu vàng",
   "Màu xanh lá",
   "Màu xanh dương",
   "Màu tím",
   "Màu hồng",
   "Màu nâu",
   "Màu đen",
   "Màu trắng"
  ]
 },
 {
  "id": "topic-clock",
  "texts": [
   "một giờ",
   "hai giờ",
   "ba giờ",
   "bốn giờ",
   "năm giờ",
   "sáu giờ",
   "bảy giờ",
   "tám giờ",
   "chín giờ",
   "mười giờ",
   "mười một giờ",
   "mười hai giờ"
  ]
 },
 {
  "id": "story-tho-yeu-me",
  "texts": [
   "Yêu mẹ",
   "Mẹ ơi con yêu mẹ",
   "Như cây yêu nắng vàng",
   "Mẹ ru con ngủ ngoan",
   "Con thơm lên má mẹ."
  ],
  "words": true
 },
 {
  "id": "story-tho-ban-moi",
  "texts": [
   "Bạn mới",
   "Hôm nay lớp có bạn",
   "Tóc xoăn, má hồng hồng",
   "Gấu chìa tay: chơi không?",
   "Bạn cười: mình chơi với!"
  ],
  "words": true
 },
 {
  "id": "story-tho-di-hoc",
  "texts": [
   "Đi học đúng giờ",
   "Reng reng chuông đồng hồ",
   "Gấu dậy rửa mặt nhanh",
   "Ăn sáng thật ngon lành",
   "Đến lớp không trễ nữa."
  ],
  "words": true
 },
 {
  "id": "story-tho-gio",
  "texts": [
   "Gió",
   "Gió ơi gió ở đâu",
   "Gió đưa diều lên cao",
   "Gió lay cành lao xao",
   "Gió hát ru bé ngủ."
  ],
  "words": true
 },
 {
  "id": "story-tho-ngoi-nha",
  "texts": [
   "Ngôi nhà của bé",
   "Nhà bé có mái đỏ",
   "Có cửa sổ xanh xanh",
   "Có bố mẹ, có anh",
   "Ai cũng thương bé lắm."
  ],
  "words": true
 },
 {
  "id": "story-tho-co-day",
  "texts": [
   "Cô dạy",
   "Cô dạy bé chào hỏi",
   "Cô dạy bé hát ca",
   "Về nhà bé khoe bà",
   "Bà khen: cháu giỏi quá!"
  ],
  "words": true
 },
 {
  "id": "story-tho-rua-tay",
  "texts": [
   "Rửa tay",
   "Xoa xoa xà phòng",
   "Kỳ kỳ ngón tay",
   "Nước mát rửa ngay",
   "Đôi tay sạch sẽ."
  ],
  "words": true
 },
 {
  "id": "story-tho-an-ngoan",
  "texts": [
   "Ăn ngoan",
   "Bát cơm thơm phức",
   "Có cá, có rau",
   "Gấu ăn thật mau",
   "Lớn nhanh khỏe mạnh."
  ],
  "words": true
 },
 {
  "id": "story-tho-chao-sang",
  "texts": [
   "Chào buổi sáng",
   "Ò ó o gà gáy",
   "Mặt trời dậy rồi kìa",
   "Gấu vươn vai một cái",
   "Chào buổi sáng, cả nhà!"
  ],
  "words": true
 },
 {
  "id": "story-tho-danh-rang",
  "texts": [
   "Đánh răng",
   "Bàn chải xinh xinh",
   "Kem thơm mát lạnh",
   "Chải lên chải xuống",
   "Răng trắng tinh tinh."
  ],
  "words": true
 },
 {
  "id": "story-tho-mua-roi",
  "texts": [
   "Mưa rơi",
   "Tí tách tí tách",
   "Mưa rơi ngoài hiên",
   "Cây xanh uống nước",
   "Hoa cười hồn nhiên."
  ],
  "words": true
 },
 {
  "id": "story-tho-bo",
  "texts": [
   "Bố của bé",
   "Bố cao thật cao",
   "Cõng bé đi chơi",
   "Bé ôm cổ bố",
   "Cười vang khắp nơi."
  ],
  "words": true
 },
 {
  "id": "story-tho-ong-trang",
  "texts": [
   "Trăng tròn",
   "Ông trăng tròn tròn",
   "Treo trên ngọn tre",
   "Gấu ngồi ngoài hè",
   "Ngắm trăng sáng quá."
  ],
  "words": true
 },
 {
  "id": "story-tho-ga-con",
  "texts": [
   "Chú gà con",
   "Chiếp chiếp chiếp chiếp",
   "Lông vàng như tơ",
   "Chạy theo mẹ gà",
   "Tìm thóc bên bờ."
  ],
  "words": true
 },
 {
  "id": "story-tho-ca-vang",
  "texts": [
   "Con cá vàng",
   "Cá vàng bơi bơi",
   "Đuôi xòe như quạt",
   "Bé thả mồi nhé",
   "Cá đớp ngon lành."
  ],
  "words": true
 },
 {
  "id": "story-tho-di-ngu",
  "texts": [
   "Đi ngủ",
   "Trăng lên cao rồi",
   "Gấu lên giường thôi",
   "Ôm gấu bông nhé",
   "Ngủ ngon, mai chơi."
  ],
  "words": true
 },
 {
  "id": "story-tho-cau-vong",
  "texts": [
   "Cầu vồng",
   "Mưa vừa tạnh xong",
   "Trời có cầu vồng",
   "Đỏ, cam, vàng, lục",
   "Lam, chàm, tím trong."
  ],
  "words": true
 },
 {
  "id": "story-tho-tap-dem",
  "texts": [
   "Bé tập đếm",
   "Một là con mèo",
   "Hai là con cá",
   "Ba là quả na",
   "Bốn là bông hoa."
  ],
  "words": true
 },
 {
  "id": "story-tho-chu-buom",
  "texts": [
   "Chú bướm vàng",
   "Chú bướm vàng ơi",
   "Bay đi đâu đấy",
   "Bay vào vườn này",
   "Hoa đang đợi đấy."
  ],
  "words": true
 },
 {
  "id": "story-tho-cat-do-choi",
  "texts": [
   "Cất đồ chơi",
   "Chơi xong rồi nhé",
   "Cất đồ chơi đi",
   "Xe vào trong hộp",
   "Gấu bông lên kệ."
  ],
  "words": true
 },
 {
  "id": "story-cd-cong-cha",
  "texts": [
   "Công cha như núi Thái Sơn",
   "Nghĩa mẹ như nước trong nguồn chảy ra",
   "Một lòng thờ mẹ kính cha",
   "Cho tròn chữ hiếu mới là đạo con."
  ],
  "words": true
 },
 {
  "id": "story-cd-bau-bi",
  "texts": [
   "Bầu ơi thương lấy bí cùng",
   "Tuy rằng khác giống nhưng chung một giàn."
  ],
  "words": true
 },
 {
  "id": "story-cd-con-kien",
  "texts": [
   "Con kiến mà leo cành đa",
   "Leo phải cành cụt leo ra leo vào",
   "Con kiến mà leo cành đào",
   "Leo phải cành cụt leo vào leo ra."
  ],
  "words": true
 },
 {
  "id": "story-cd-trau-oi",
  "texts": [
   "Trâu ơi ta bảo trâu này",
   "Trâu ra ngoài ruộng trâu cày với ta",
   "Cấy cày vốn nghiệp nông gia",
   "Ta đây trâu đấy ai mà quản công."
  ],
  "words": true
 },
 {
  "id": "story-cd-con-co",
  "texts": [
   "Con cò mà đi ăn đêm",
   "Đậu phải cành mềm lộn cổ xuống ao",
   "Ông ơi ông vớt tôi nao",
   "Tôi có lòng nào ông hãy xáo măng",
   "Có xáo thì xáo nước trong",
   "Đừng xáo nước đục đau lòng cò con."
  ],
  "words": true
 },
 {
  "id": "story-dd-thang-bom",
  "texts": [
   "Thằng Bờm",
   "Thằng Bờm có cái quạt mo",
   "Phú ông xin đổi ba bò chín trâu",
   "Bờm rằng Bờm chẳng lấy trâu",
   "Phú ông xin đổi một xâu cá mè",
   "Bờm rằng Bờm chẳng lấy mè",
   "Phú ông xin đổi một bè gỗ lim",
   "Bờm rằng Bờm chẳng lấy lim",
   "Phú ông xin đổi đôi chim đồi mồi",
   "Bờm rằng Bờm chẳng lấy mồi",
   "Phú ông xin đổi nắm xôi, Bờm cười."
  ],
  "words": true
 },
 {
  "id": "story-dd-cai-bong",
  "texts": [
   "Cái bống",
   "Cái bống là cái bống bang",
   "Khéo sảy khéo sàng cho mẹ nấu cơm",
   "Mẹ bống đi chợ đường trơn",
   "Bống ra gánh đỡ chạy cơn mưa ròng."
  ],
  "words": true
 },
 {
  "id": "story-dd-ong-giang",
  "texts": [
   "Ông giẳng ông giăng",
   "Xuống chơi với tôi",
   "Có bầu có bạn",
   "Có ván cơm xôi",
   "Có nồi cơm nếp",
   "Có nệp bánh chưng",
   "Có lưng hũ rượu",
   "Có chiếu bám đu."
  ],
  "words": true
 },
 {
  "id": "story-dd-con-ga",
  "texts": [
   "Con gà cục tác lá chanh",
   "Con lợn ủn ỉn mua hành cho tôi",
   "Con chó khóc đứng khóc ngồi",
   "Mẹ ơi đi chợ mua tôi đồng riềng."
  ],
  "words": true
 },
 {
  "id": "story-dd-dung-dang",
  "texts": [
   "Dung dăng dung dẻ",
   "Dắt trẻ đi chơi",
   "Đến cửa nhà trời",
   "Lạy cậu lạy mợ",
   "Cho cháu về quê",
   "Cho dê đi học",
   "Cho cóc ở nhà",
   "Cho gà bới bếp",
   "Ngồi xệp xuống đây."
  ],
  "words": true
 },
 {
  "id": "story-dd-con-meo",
  "texts": [
   "Con mèo mà trèo cây cau",
   "Hỏi thăm chú chuột đi đâu vắng nhà",
   "Chú chuột đi chợ đường xa",
   "Mua mắm mua muối giỗ cha con mèo."
  ],
  "words": true
 },
 {
  "id": "story-dd-keo-cua",
  "texts": [
   "Kéo cưa lừa xẻ",
   "Ông thợ nào khỏe",
   "Về ăn cơm vua",
   "Ông thợ nào thua",
   "Về bú tí mẹ."
  ],
  "words": true
 },
 {
  "id": "story-dd-nu-na",
  "texts": [
   "Nu na nu nống",
   "Cái cống nằm trong",
   "Cái ong nằm ngoài",
   "Củ khoai chấm mật",
   "Phật ngồi phật khóc",
   "Con cóc nhảy ra",
   "Con gà ú ụ",
   "Nhà mụ thổi xôi",
   "Nhà tôi nấu chè",
   "Tay xòe chân rụt."
  ],
  "words": true
 },
 {
  "id": "story-dd-ba-cong",
  "texts": [
   "Bà còng đi chợ",
   "Bà còng đi chợ trời mưa",
   "Cái tôm cái tép đi đưa bà còng",
   "Đưa bà qua quãng đường đông",
   "Đưa bà vào tận ngõ trong nhà bà."
  ],
  "words": true
 },
 {
  "id": "story-tr-rua-tho",
  "texts": [
   "Rùa và Thỏ",
   "Thỏ chạy rất nhanh nên hay chê Rùa chậm chạp.",
   "Rùa bảo: mình thi chạy nhé!",
   "Thỏ chạy vèo một cái đã xa, rồi nằm dưới gốc cây ngủ khì.",
   "Rùa cứ chậm mà chắc, bò mãi, bò mãi không nghỉ.",
   "Khi Thỏ tỉnh dậy, Rùa đã tới đích rồi.",
   "Chăm chỉ từng chút sẽ thành công."
  ],
  "words": true
 },
 {
  "id": "story-tr-gau-mat",
  "texts": [
   "Gấu con và hũ mật",
   "Gấu con tìm được một hũ mật ong thơm lừng.",
   "Gấu định ăn một mình thì thấy bạn Sóc đang đói bụng.",
   "Gấu nghĩ một lát, rồi chia cho Sóc một nửa.",
   "Hai bạn cùng ăn, cùng cười, mật ngọt hơn hẳn.",
   "Chia sẻ với bạn bè làm mình vui hơn đấy!"
  ],
  "words": true
 },
 {
  "id": "story-tr-kien-bo-cau",
  "texts": [
   "Kiến và Bồ Câu",
   "Một chú Kiến bị rơi xuống suối.",
   "Bồ Câu thả một chiếc lá, Kiến bám vào và lên bờ.",
   "Hôm sau, có người định bắt Bồ Câu.",
   "Kiến bò tới cắn vào chân người đó, Bồ Câu bay thoát.",
   "Giúp bạn hôm nay, mai bạn giúp lại mình."
  ],
  "words": true
 },
 {
  "id": "story-tr-ngoi-sao",
  "texts": [
   "Ngôi sao nhỏ đi ngủ",
   "Trên trời có một ngôi sao nhỏ không chịu đi ngủ.",
   "Mặt Trăng hỏi: sao con còn thức?",
   "Sao nhỏ nói: con sợ ngủ rồi không ai chiếu sáng.",
   "Mặt Trăng cười: con ngủ đi, sáng mai Mặt Trời sẽ dậy.",
   "Sao nhỏ nhắm mắt, mơ một giấc mơ thật đẹp."
  ],
  "words": true
 }
]'''
pathlib.Path('voice-texts.json').write_text(TEXTS, encoding='utf8')
jobs = json.loads(TEXTS)
print(f'✓ {len(jobs)} nhóm, {sum(len(j["texts"]) for j in jobs)} câu')

## 3. Chọn giọng

**Cách A — Giọng của mẹ/bố (khuyên dùng, ổn định nhất):** ghi âm bằng điện thoại 5–10 giây, giọng vui, rõ, phòng yên tĩnh, đọc đúng câu:

> *Chào con yêu! Hôm nay cô cùng con đọc thơ, đếm số và tìm các bạn con vật nhé. Con sẵn sàng chưa nào?*

Đặt `CACH = "ghi_am"` rồi chạy ô bên dưới, chọn file ghi âm để tải lên (.m4a, .mp3, .wav đều được).

**Cách B — Để máy tự thiết kế giọng cô giáo:** đặt `CACH = "thiet_ke"`. OmniVoice chủ yếu học thiết kế giọng bằng tiếng Anh/Trung nên với tiếng Việt đôi khi chưa ổn — nghe thử ở ô 4, chưa hay thì đổi `MO_TA` hoặc chạy lại để ra giọng khác.

In [ ]:
#@title 3. Chọn giọng
CACH = "ghi_am"  #@param ["ghi_am", "thiet_ke"]
LOI_TRONG_GHI_AM = "Chào con yêu! Hôm nay cô cùng con đọc thơ, đếm số và tìm các bạn con vật nhé. Con sẵn sàng chưa nào?"  #@param {type:"string"}
MO_TA = "female, young adult, high pitch"  #@param {type:"string"}
MO_HINH = "k2-fsa/OmniVoice"  #@param ["k2-fsa/OmniVoice", "termanteus/omnivoice-vietnamese"]
TOC_DO_THE = 0.85  #@param {type:"slider", min:0.6, max:1.1, step:0.05}
TOC_DO_THO = 0.8  #@param {type:"slider", min:0.6, max:1.1, step:0.05}

import subprocess, shutil, pathlib
VOICE_ARGS = ["--omni-model", MO_HINH, "--speed", str(TOC_DO_THE), "--story-speed", str(TOC_DO_THO), "--new-voice"]
if CACH == "ghi_am":
    from google.colab import files
    up = files.upload()
    src = next(iter(up))
    subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-i", src, "-ac", "1", "-ar", "24000", "ref.wav"], check=True)
    VOICE_ARGS += ["--ref-audio", "ref.wav", "--ref-text", LOI_TRONG_GHI_AM]
    print("✓ Đã nhận ghi âm:", src)
else:
    VOICE_ARGS += ["--instruct", MO_TA]
    print("✓ Sẽ thiết kế giọng:", MO_TA)
shutil.rmtree(".voice-cache", ignore_errors=True)  # đổi giọng thì đọc lại từ đầu

In [ ]:
#@title 4. Nghe thử một bài thơ (1–2 phút)
import subprocess, sys
from IPython.display import Audio, display
r = subprocess.run([sys.executable, "build_voice_pack.py", "--engine", "omnivoice", "--texts", "voice-texts.json",
                    "--out", "audio", "--cache", ".voice-cache", "--only", "story-tho-yeu-me", *VOICE_ARGS], capture_output=True, text=True)
print(r.stdout[-1500:], r.stderr[-1500:])
display(Audio("audio/story-tho-yeu-me.mp3"))
print("Hay rồi thì chạy ô 5. Chưa hay: sửa ô 3 rồi chạy lại ô 3 và ô 4.")

In [ ]:
#@title 5. Đọc toàn bộ app (khoảng 10–20 phút trên T4)
import subprocess, sys
args = [a for a in VOICE_ARGS if a != "--new-voice"]  # giữ đúng giọng vừa nghe thử
p = subprocess.Popen([sys.executable, "build_voice_pack.py", "--engine", "omnivoice", "--texts", "voice-texts.json",
                      "--out", "audio", "--cache", ".voice-cache", *args], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in p.stdout:
    print(line, end="")
p.wait()
print("✓ Xong" if p.returncode == 0 else "✗ Có lỗi — chạy lại ô này, phần đã xong được giữ lại")

In [ ]:
#@title 6. Tải gói giọng về
!cd audio && rm -f ../gau-voice-pack.zip && zip -qr ../gau-voice-pack.zip . && cd .. && ls -la gau-voice-pack.zip
from google.colab import files
files.download("gau-voice-pack.zip")
print("Gửi file gau-voice-pack.zip cho Claude để cập nhật app.")